[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/halla-ai/deepnlp-2026/blob/main/notebooks/week-06.ipynb)

# 6주차 실습: 멀티모달 표현의 결합

**목표.** 제주 공개 사진 5장으로 세 가지를 해 본다.

1. **CLIP 제로샷 분류**: 사진과 후보 문장의 유사도 히트맵을 그리고, 라벨 언어(영어/한국어)가 결과를 어떻게 바꾸는지 본다
2. **SmolVLM 질의응답**: 사진을 언어모형에 넣고 질문해 답을 얻는다. **사진만 바꿔** 답이 사진을 따라 움직이는지(근거), 한국어 질문이 어떻게 깨지는지(언어 편향) 확인한다
3. **텍스트 전용 모델과 비교, 시각 토큰 세기**: 같은 질문을 사진 없이 텍스트 전용 모델에 주면 무엇이 나오는지 보고, 사진 1장이 문맥에서 몇 토큰을 쓰는지와 그 수를 정하는 것이 무엇인지 잰다

GPU는 필요 없다. 노트북이 실행 시점에 사진과 모델을 내려받는다. 사진은 위키미디어 공용과 공공누리의 제주 공개 사진이고, 출처와 라이선스는 강의 노트 6주차 [사진 출처] 표에 있다.

## 0. 준비

아래 셀을 실행해 필요한 라이브러리를 설치한다. GPU는 필요 없다.

> transformers가 5.x면 시각 모델 클래스가 `AutoModelForImageTextToText`다. 4.x의 `AutoModelForVision2Seq`는 이름이 바뀌었다. 이 노트북은 5.x 클래스를 쓴다.

In [ ]:
# 필요한 것 설치 (Colab에서 한 번만)
!pip -q install -U transformers
!pip -q install torch torchvision pillow

## 1. 먼저 그냥 실행해 보기

아래 셀들을 위에서부터 차례로 실행하세요. 아무것도 고치지 않아도 끝까지 돌아갑니다.

### 1-1. 제주 공개 사진 5장 내려받기

위키미디어 공용(및 공공누리)에서 라이선스가 확인된 제주 사진 5장을 내려받는다. 강의 노트의 실측과 같은 사진이다.

In [ ]:
import urllib.request
import time

IMAGES = {
    # 파일명: (다운로드 주소, 위키미디어 공용 원본 제목)
    "seongsan-sunrise": (
        "https://upload.wikimedia.org/wikipedia/commons/thumb/f/f4/Sunrise_with_Seongsan_Ilchulbong.jpg/960px-Sunrise_with_Seongsan_Ilchulbong.jpg",
        "Sunrise with Seongsan Ilchulbong",
    ),
    "seongsan-carpark": (
        "https://upload.wikimedia.org/wikipedia/commons/thumb/b/b0/Sunrise_peak_carpark_-_panoramio.jpg/960px-Sunrise_peak_carpark_-_panoramio.jpg",
        "Sunrise peak carpark - panoramio",
    ),
    "hallasan-yeongsil": (
        "https://upload.wikimedia.org/wikipedia/commons/thumb/e/e5/Wooden_staircase_along_Yeongsil_Trail_with_the_mountains_of_Hallasan_Park_Jeju_Island_South_Korea.jpg/960px-Wooden_staircase_along_Yeongsil_Trail_with_the_mountains_of_Hallasan_Park_Jeju_Island_South_Korea.jpg",
        "Wooden staircase along Yeongsil Trail",
    ),
    "manjanggul": (
        "https://upload.wikimedia.org/wikipedia/commons/8/80/Lava_Manjanggul_Cave.jpg",
        "Lava Manjanggul Cave",
    ),
    "udo": (
        "https://upload.wikimedia.org/wikipedia/commons/thumb/0/04/Udo%2C_Jeju_Province%2C_South_Korea_02.jpg/960px-Udo%2C_Jeju_Province%2C_South_Korea_02.jpg",
        "Udo, Jeju Province, South Korea 02",
    ),
}

import os
os.makedirs("week06-imgs", exist_ok=True)

def fetch(url, tries=4):
    # 위키미디어는 짧은 시간에 요청이 몰리면 429를 돌려준다. 기다렸다 다시 요청한다.
    for k in range(tries):
        req = urllib.request.Request(url, headers={"User-Agent": "deepnlp-course/1.0"})
        try:
            with urllib.request.urlopen(req, timeout=60) as r:
                return r.read()
        except urllib.error.HTTPError as e:
            if e.code == 429 and k < tries - 1:
                wait = 15 * (k + 1)
                print(f"  429, {wait}초 후 재시도 ({k + 1}/{tries - 1})")
                time.sleep(wait)
            else:
                raise

for name, (url, _title) in IMAGES.items():
    path = f"week06-imgs/{name}.jpg"
    if not os.path.exists(path):
        data = fetch(url)
        with open(path, "wb") as f:
            f.write(data)
    print(f"{path}: {os.path.getsize(path) // 1024} KB")

### 1-2. 사진 한눈에 보기

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image

fig, axes = plt.subplots(1, 5, figsize=(15, 3))
for ax, (name, (_url, title)) in zip(axes, IMAGES.items()):
    img = Image.open(f"week06-imgs/{name}.jpg").convert("RGB")
    ax.imshow(img)
    ax.set_title(name, fontsize=9)
    ax.axis("off")
plt.tight_layout()
plt.show()

### 1-3. CLIP 제로샷 - 사진과 문장의 유사도

CLIP은 이미지 인코더와 텍스트 인코더 **둘**을 가진다. 이미지 인코더는 ViT-B/32다. 사진을 224x224로 맞춘 뒤 32x32 픽셀 조각(패치) 7x7=49개로 잘라 조각 하나를 토큰처럼 읽는다. 두 인코더가 각자 벡터를 만들고, 벡터를 길이 1로 정규화한 뒤 내적하면 코사인 유사도가 나온다.

사진 5장 x 후보 문장 5개의 유사도 행렬을 만들어 본다. 각 행에서 가장 큰 값이 모델이 고른 문장이다.

In [ ]:
import torch
from transformers import CLIPModel, CLIPProcessor

clip = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").eval()
clip_proc = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")

print(f"logit_scale: {clip.logit_scale.exp().item():.1f}")

def clip_similarity(image_paths, texts):
    # 이미지와 문장을 각자 인코딩해 정규화한 뒤 내적한다
    imgs = [Image.open(p).convert("RGB") for p in image_paths]
    with torch.no_grad():
        img_inputs = clip_proc(images=imgs, return_tensors="pt")
        img_emb = clip.get_image_features(pixel_values=img_inputs["pixel_values"])
        if hasattr(img_emb, "pooler_output"):
            img_emb = img_emb.pooler_output
        txt_inputs = clip_proc(text=texts, return_tensors="pt", padding=True)
        txt_emb = clip.get_text_features(
            input_ids=txt_inputs["input_ids"], attention_mask=txt_inputs["attention_mask"]
        )
        if hasattr(txt_emb, "pooler_output"):
            txt_emb = txt_emb.pooler_output
        img_emb = img_emb / img_emb.norm(dim=-1, keepdim=True)
        txt_emb = txt_emb / txt_emb.norm(dim=-1, keepdim=True)
        return (img_emb @ txt_emb.T).tolist()

# 동작 확인: 첫 사진 1장, 짧은 영어 라벨 5개
labels_en_coarse = [
    "a photo of a sunrise peak",
    "a photo of a parking lot",
    "a photo of a hiking trail",
    "a photo of a lava tube cave",
    "a photo of a coastal island",
]
paths = [f"week06-imgs/{name}.jpg" for name in IMAGES]
sim = clip_similarity(paths, labels_en_coarse)
print(f"\n{IMAGES['seongsan-sunrise'][1]} 행:")
for lab, v in zip(labels_en_coarse, sim[0]):
    print(f"  {v:+.3f}  {lab}")

### 1-4. 라벨 언어가 결과를 바꾼다 - 유사도 히트맵

같은 사진 5장에 라벨 조합을 네 가지로 바꿔가며 제로샷 분류를 돌린다.

- **영어 (짧은)**: `a photo of a ...` 뒤에 짧은 영어 라벨
- **영어 (장면 묘사)**: 장면을 묘사하는 긴 영어 라벨
- **한국어**: `... 사진` 형식의 한국어 라벨
- **한국어 + 영어 섞기**: 한국어 라벨에 영어를 괄호로 붙임

라벨 조합마다 **각 사진이 고른 라벨**도 출력한다. 한국어 라벨에서 여러 사진이 같은 라벨로 쏠리는지 보자.

히트맵에서 행: 사진, 열: 후보 라벨이다. 가장 진한 칸이 모델이 고른 라벨이고, **대각선이 진하면 정답을 골라낸 것**이다.

In [ ]:
LABEL_SETS = {
    "en_coarse": labels_en_coarse,
    "en_scenario": [
        "a photo of a sunrise over a volcanic crater by the sea",
        "a photo of a parking lot full of tour buses",
        "a photo of a wooden staircase on a forest hiking trail",
        "a photo of a dark lava tube cave interior",
        "a photo of green sea cliffs and a small island",
    ],
    "ko": [
        "성산일출봉 사진",
        "주차장 사진",
        "등산로 사진",
        "용암동굴 사진",
        "해안 섬 사진",
    ],
    "ko_english_mix": [
        "성산일출봉 (Seongsan Ilchulbong) sunrise peak photo",
        "주차장 (parking lot) photo",
        "등산로 (hiking trail) photo",
        "용암동굴 (lava tube cave) photo",
        "해안 섬 (coastal island) photo",
    ],
}

img_names = list(IMAGES)
all_sims = {}
for set_name, labels in LABEL_SETS.items():
    all_sims[set_name] = clip_similarity(paths, labels)
    correct = sum(1 for i in range(len(paths)) if max(range(len(labels)), key=lambda j: all_sims[set_name][i][j]) == i)
    print(f"{set_name}: 대각선 정답 {correct}/{len(paths)}")
    for i, name in enumerate(img_names):
        j = max(range(len(labels)), key=lambda j: all_sims[set_name][i][j])
        print(f"    {name:<18} -> {labels[j]}{'' if j == i else '  (오답)'}")

fig, axes = plt.subplots(1, 4, figsize=(19, 4.2))
short = {"en_coarse": "EN short", "en_scenario": "EN scenario", "ko": "KO", "ko_english_mix": "KO+EN mix"}
for ax, (set_name, labels) in zip(axes, LABEL_SETS.items()):
    data = all_sims[set_name]
    ax.imshow(data, cmap="viridis", vmin=0.1, vmax=0.35)
    ax.set_xticks(range(len(labels)))
    ax.set_xticklabels(range(1, len(labels) + 1), fontsize=8)
    ax.set_yticks(range(len(img_names)))
    ax.set_yticklabels(img_names, fontsize=8)
    for i in range(len(paths)):
        for j in range(len(labels)):
            ax.text(j, i, f"{data[i][j]:.3f}", ha="center", va="center", fontsize=6.5,
                    color="white" if data[i][j] < 0.3 else "black")
    ax.set_title(short[set_name], fontsize=10)
plt.suptitle("CLIP zero-shot cosine similarity (rows: images, cols: labels)", fontsize=11)
plt.tight_layout()
plt.show()

print("열 라벨 번호:")
for set_name, labels in LABEL_SETS.items():
    print(f"  {set_name}: " + ", ".join(f"{j+1}={l}" for j, l in enumerate(labels)))

### 1-5. 한국어 라벨은 토큰에서 어떻게 되는가

한국어 라벨이 무너지는 원인을 토큰에서 확인한다. CLIP의 토크나이저는 한국어 글자를 **바이트 조각**으로 자른다. 한글 한 글자는 UTF-8로 3바이트라서 보통 1~2개 조각이 된다. CLIP의 텍스트 인코더는 한 문장에 토큰을 77개까지만 받으므로, 한국어는 같은 길이의 영어보다 이 한도에 빨리 닿는다. 토큰 자체는 만들어지므로 무너지는 원인은 토큰화가 아니라, 그 토큰들이 사진 내용과 짝지어져 학습된 일이 드물다는 것(학습 데이터가 영어 위주)이다. 그래서 다섯 한국어 라벨의 벡터가 서로 거의 구별되지 않는다.

In [ ]:
ko_text = "성산일출봉 사진"
ko_inputs = clip_proc(text=[ko_text], return_tensors="pt")
ko_ids = ko_inputs["input_ids"][0].tolist()
en_text = "a photo of a sunrise peak"
en_inputs = clip_proc(text=[en_text], return_tensors="pt")
en_ids = en_inputs["input_ids"][0].tolist()

print(f"한국어 '{ko_text}' -> 토큰 {len(ko_ids)}개: {ko_ids}")
print(f"영어   '{en_text}' -> 토큰 {len(en_ids)}개: {en_ids}")
print(f"'성' 한 글자 -> 조각 {len(clip_proc.tokenizer('성', add_special_tokens=False).input_ids)}개: {clip_proc.tokenizer.tokenize('성')}")
print(f"어휘 크기: {clip.config.text_config.vocab_size} (바이트 단위 BPE 어휘 전체)")
print()
print("한국어는 글자가 바이트 조각으로 분해된다. 토큰은 만들어지지만,")
print("그 토큰들이 사진 내용과 짝지어져 학습된 일이 드물어 라벨끼리 벡터가 거의 구별되지 않는다.")

### 1-6. SmolVLM 질의응답 - 사진을 언어모형에 넣기

이제 생성형이다. SmolVLM-256M-Instruct는 사진을 **시각 토큰**으로 바꿔 언어모형의 문맥에 끼워 넣는다.

시각 토큰 수는 **타일 분할** 설정이 크게 바꾼다.

- 분할을 켜면(모델 기본값) 사진을 긴 변 2048로 키운 뒤 512x512 타일로 자르고, 전체 사진 축소본 1장을 더한다. 타일 1장이 64토큰이므로 960x640 사진은 타일 12장 + 1장 = 13장 x 64 = **832토큰**이다
- 분할을 끄면 전체 사진 1장만 넣어 **64토큰**이다

분할을 켜면 CPU에서 답 하나에 수십 초가 걸린다. 그래서 이 노트북은 **분할을 끄고(`SPLIT = False`) 답을 생성한다.** 분할을 켰을 때와의 차이는 1-10에서 직접 잰다.

절차는 세 단계다.

1. 채팅 템플릿으로 대화 형식 프롬프트를 조립한다 (이미지 자리는 표시자로 둔다)
2. 사진과 프롬프트를 프로세서에 넣어 모델 입력을 만든다
3. `generate()`로 답을 생성한다

In [ ]:
from transformers import AutoProcessor, AutoModelForImageTextToText

vlm_name = "HuggingFaceTB/SmolVLM-256M-Instruct"
vlm_processor = AutoProcessor.from_pretrained(vlm_name)
vlm = AutoModelForImageTextToText.from_pretrained(vlm_name).eval()

SPLIT = False  # 타일 분할. False면 사진 1장이 시각 토큰 64개가 되어 CPU에서도 빠르게 답한다
vlm_processor.image_processor.do_image_splitting = SPLIT

total_params = sum(p.numel() for p in vlm.parameters())
print(f"전체 파라미터: {total_params:,}")

def build_inputs(img, question, split=None):
    # 채팅 템플릿으로 프롬프트를 만들고 사진과 함께 모델 입력으로 바꾼다
    messages = [{"role": "user", "content": [{"type": "image"}, {"type": "text", "text": question}]}]
    prompt = vlm_processor.apply_chat_template(messages, add_generation_prompt=True)
    kw = {} if split is None else {"do_image_splitting": split}
    return vlm_processor(images=img, text=prompt, return_tensors="pt", **kw)

def vlm_ask(image_path, question, max_new_tokens=48):
    # 사진과 질문을 넣어 답을 생성한다 (탐욕 디코딩이라 매번 같은 답)
    img = Image.open(image_path).convert("RGB")
    inputs = build_inputs(img, question)
    with torch.no_grad():
        out = vlm.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    text = vlm_processor.batch_decode(out, skip_special_tokens=True)[0]
    return text.split("Assistant:")[-1].strip()

def visual_token_count(img, split):
    # 입력 전체 토큰 수와 그중 시각 토큰(이미지 표시자) 수를 센다
    ids = build_inputs(img, "Describe this image.", split=split)["input_ids"][0].tolist()
    return len(ids), sum(1 for t in ids if t == vlm_processor.image_token_id)

first = "week06-imgs/seongsan-sunrise.jpg"
img0 = Image.open(first).convert("RGB")
for split in [True, False]:
    n_total, n_vis = visual_token_count(img0, split)
    print(f"분할 {'켬' if split else '끔'}: 입력 토큰 {n_total}개 중 시각 토큰 {n_vis}개 ({n_vis / n_total:.0%})")

print("\n동작 확인:")
q = "Describe this image in one sentence."
print(f"Q: {q}")
print(f"A: {vlm_ask(first, q)}")

### 1-7. 사진만 바꿔 보기 - 답이 사진을 따라 움직이는가

질문을 고정하고 **사진만** 다섯 장으로 바꿔 본다. 답이 사진 내용을 따라 바뀌면 답이 이미지에 근거(grounded) 있다는 뜻이다.

In [ ]:
QUESTION_EN = "Describe this image in one sentence."
print(f"질문(고정): {QUESTION_EN}\n")
for name in IMAGES:
    path = f"week06-imgs/{name}.jpg"
    answer = vlm_ask(path, QUESTION_EN)
    print(f"[{name}]")
    print(f"  {answer}\n")

### 1-8. 한국어 질문은 어떻게 깨지는가

같은 사진들에 한국어 질문을 넣어 본다. 답이 영어로 나오거나, 질문과 상관없는 내용이 나오거나, 깨진 한글이 나온다. 영어 질문일 때보다 답이 사진에서 멀어지는 경우도 찾아보자.

원인은 모델 크기만이 아니라 **학습 데이터의 언어 분포**다. 256M 모델의 지시 학습 데이터는 영어 위주라 한국어 지시 뒤의 다음 토큰 확률이 무너진다.

In [ ]:
QUESTION_KO = "이 이미지를 한 문장으로 묘사하라."
print(f"Q: {QUESTION_KO}\n")
for name in IMAGES:
    print(f"[{name}] {vlm_ask(f'week06-imgs/{name}.jpg', QUESTION_KO)}")

q2 = "여기가 어디인가? 주된 피사체가 무엇인가?"
print(f"\nQ: {q2}")
print(f"[seongsan-sunrise] {vlm_ask(first, q2)}")

### 1-9. 텍스트 전용 모델에게 같은 질문을 하면

온라인 학습활동의 비교다. 사진을 받을 수 없는 텍스트 전용 모델(Qwen3-0.6B-Base, 4·5주차와 같은 모델)에게 같은 질문을 준다. 사진이 없으니 모델은 **문맥만 보고 그럴듯한 장면을 지어낸다.** 답이 사진과 아무 관계가 없다는 것을 확인하자.

프롬프트 형식은 `Question: ...\nAnswer:`이고 탐욕 디코딩이라 매번 같은 답이 나온다.

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

txt_name = "Qwen/Qwen3-0.6B-Base"
txt_tok = AutoTokenizer.from_pretrained(txt_name)
txt_model = AutoModelForCausalLM.from_pretrained(txt_name).eval()

def text_only_ask(question, max_new_tokens=40):
    ids = txt_tok(f"Question: {question}\nAnswer:", return_tensors="pt").input_ids
    with torch.no_grad():
        out = txt_model.generate(ids, max_new_tokens=max_new_tokens, do_sample=False, pad_token_id=txt_tok.eos_token_id)
    return txt_tok.decode(out[0][ids.shape[1]:], skip_special_tokens=True).strip()

for q in ["Describe this image in one sentence.", "What is in this photo?", "사진 속에 나무가 몇 그루인가?"]:
    print(f"Q: {q}")
    print(f"  텍스트 전용: {text_only_ask(q)}")
    print(f"  SmolVLM (성산 사진): {vlm_ask(first, q)}\n")

### 1-10. 시각 토큰 수는 무엇이 정하나 - 크기, 비율, 분할

학습활동 [실무]의 지연·비용 보고를 위한 측정이다. 같은 사진을 여러 크기와 가로세로 비율로 바꿔 시각 토큰 수를 센다. **해상도를 올리면 토큰이 늘어날까?** 먼저 예상해 보고 실행하자.

마지막 줄은 분할을 켰을 때와 껐을 때 **첫 토큰이 나오기까지 걸린 시간**이다. 사진을 읽고 문맥을 처리하는 비용이 이 시간에 들어 있다. Colab 무료 CPU에서는 이보다 오래 걸릴 수 있다.

In [ ]:
import time

print(f"{'크기':<12} {'분할 켬':>8} {'분할 끔':>8}")
for w, h in [(240, 160), (960, 640), (1920, 1280), (640, 640), (1280, 320)]:
    im = img0.resize((w, h))
    on, off = visual_token_count(im, True)[1], visual_token_count(im, False)[1]
    print(f"{w}x{h:<7} {on:>8} {off:>8}")

for split in [True, False]:
    inputs = build_inputs(img0, "Describe this image in one sentence.", split=split)
    t = time.time()
    with torch.no_grad():
        vlm.generate(**inputs, max_new_tokens=1, do_sample=False)
    print(f"분할 {'켬' if split else '끔'}: 입력 {inputs['input_ids'].shape[1]}토큰, 첫 토큰까지 {time.time() - t:.1f}초")

## 2. 한 지점만 바꿔 보기 - 입력 이미지를 바꾼다

아래 셀의 `# TODO`로 표시된 **한 곳만** 바꾸고 다시 실행하세요.

> 규칙: `MY_IMAGE`를 다른 사진 이름으로 바꾼다. 후보는 `seongsan-sunrise`, `seongsan-carpark`, `hallasan-yeongsil`, `manjanggul`, `udo` 다섯 가지다. 바꾼 뒤 두 가지를 기록한다. (1) 같은 질문인데 답이 사진을 따라 바뀌는가. (2) 바꾼 사진의 특징이 답에 어떻게 반영되는가(또는 어두운 사진에서 답이 어떻게 무너지는가). 이것이 학습활동 [구현]의 기록이다.

In [ ]:
# TODO: 사진 이름을 바꾼다. 후보: seongsan-sunrise / seongsan-carpark / hallasan-yeongsil / manjanggul / udo
MY_IMAGE = "seongsan-sunrise"

# 아래는 그대로 둡니다
path = f"week06-imgs/{MY_IMAGE}.jpg"
print(f"사진: {MY_IMAGE}")
print(f"\n[영어 질문]")
q = "Describe this image in one sentence."
a = vlm_ask(path, q)
print(f"  Q: {q}")
print(f"  A: {a}")
print(f"\n[한국어 질문]")
q_ko = "이 이미지를 한 문장으로 묘사하라."
a_ko = vlm_ask(path, q_ko)
print(f"  Q: {q_ko}")
print(f"  A: {a_ko}")
print(f"\n[제로샷 - 이 사진은 다섯 라벨 중 어디에 가까운가]")
sim_one = clip_similarity([path], LABEL_SETS["en_coarse"])[0]
best = max(range(5), key=lambda j: sim_one[j])
for j, (lab, v) in enumerate(zip(LABEL_SETS["en_coarse"], sim_one)):
    mark = "  <-- 모델의 선택" if j == best else ""
    print(f"  {v:+.3f}  {lab}{mark}")

## 3. 정리 그래프

네 라벨 조합의 대각선 정답 수를 막대로 그린다. 라벨 언어가 제로샷을 어떻게 바꾸는지 한눈에 본다.

In [ ]:
accs = {}
for set_name, labels in LABEL_SETS.items():
    accs[set_name] = sum(
        1 for i in range(len(paths)) if max(range(len(labels)), key=lambda j: all_sims[set_name][i][j]) == i
    ) / len(paths)

plt.figure(figsize=(7, 4))
names = [short[n] for n in LABEL_SETS]
vals = [accs[n] for n in LABEL_SETS]
bars = plt.bar(names, vals, color=["#4263eb", "#2f9e44", "#e03131", "#f08c00"])
for bar, v in zip(bars, vals):
    plt.text(bar.get_x() + bar.get_width() / 2, v + 0.03, f"{v:.0%}", ha="center", fontsize=11)
plt.ylim(0, 1.15)
plt.ylabel("zero-shot diagonal accuracy")
plt.title("Label language changes CLIP zero-shot accuracy (5 Jeju photos)")
plt.tight_layout()
plt.show()

## 4. 확인 질문

1. 라벨 조합 네 가지의 대각선 정답 수는 각각 몇이었나요? 한국어 라벨에서 사진들은 어느 라벨로 쏠렸나요?
2. 영어 짧은 라벨에서 틀린 사진은 무엇이고, 두 후보의 유사도 차이는 얼마였나요? logit_scale 100을 곱하면 확률은 어떻게 갈리나요?
3. 1-7에서 사진을 바꿨을 때 답이 사진을 따라 바뀌었나요? 답에 사진에 없는 것이 들어간 경우가 있었나요?
4. 1-9에서 텍스트 전용 모델의 답은 사진과 어떤 관계였나요? 5주차의 어떤 개념과 이어지나요?
5. 1-10에서 해상도를 올리면 시각 토큰이 늘었나요? 토큰 수를 실제로 바꾼 것은 무엇이었나요?
6. 한국어 질문의 답은 어떤 모양이었나요? 질문 언어와 답 언어의 관계에서 관찰한 것을 한 문장으로 써 보세요.

답은 아래 셀에 글로 적으면 됩니다. 코드가 아니어도 됩니다.

*(여기에 답을 적으세요)*

## 5. 제출

1. 상단 메뉴 **파일 > .ipynb 다운로드** 로 이 노트북을 내려받습니다
2. [저장소](https://github.com/halla-ai/deepnlp-2026)의 `assignments/week-06/<내 학번>/` 에 업로드합니다
3. Pull Request를 엽니다

자세한 방법은 강의 사이트의 **과제 제출** 문서에 있습니다.

---

**막혔나요?** 오류 메시지의 마지막 줄을 먼저 읽어 보세요. 그래도 안 되면 AI Professor 튜터에게 묻고, 그래도 막히면 저장소 Issues에 남기세요.